# Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.impute import SimpleImputer

from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline as ImbPipeline

import xgboost as xgb

In [ ]:
df = pd.read_csv('../ml/df_model.csv', sep=';', dtype={'Code_INSEE': str})
display(df.head(5))
print(f"Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

print(f"\nDistribution de la cible (vote politique) :")
print(df['Résultat'].value_counts())
print(f"\nProportions :")
print(df['Résultat'].value_counts(normalize=True).round(4) * 100)

print('\nColonnes :')
print(df.columns)

# Feature engineering

### Ordinal encoding de la variable cible

In [ ]:
groups = {'gauche': 0, 'centre': 1, 'droite': 2}
df['Résultat'] = df['Résultat'].replace(groups).astype(int)
display(df)

### Autres colonnes

In [ ]:
# On supprime 'homme seul' et 'femme seule' (déjà inclus dans 'personne seule')
df = df.drop(columns=['Femme seule', 'Homme seul'])


# On transforme 'population_active' en un booléen 'grande_ville'
df['Grande ville'] = df['Population_active'].map(lambda x: True if x >= 10000 else False)
df = df.drop(columns=['Population_active'])


# On ajoute un booléen pour savoir si la commune possède une majorité de personnes actives (ou garder la proportion brute)
df['Total actifs'] = df['Agriculteurs'] + df['Artisans'] + df['Cadres'] + df['Intermédiaires'] + df['Employés'] + df['Ouvriers']
df['Majorité actifs'] = df['Total actifs'].map(lambda x: True if x > 50 else False)
df = df.drop(columns=['Total actifs'])


# On remplace les proportions de chaque catégorie socio-professionnelle par la CSP majoritaire
cols_actifs = ['Agriculteurs', 'Artisans', 'Cadres', 'Intermédiaires', 'Employés', 'Ouvriers']
mask_ages = df[cols_actifs].notna().any(axis=1)
df.loc[mask_ages, 'CSP majoritaire'] = df.loc[mask_ages, cols_actifs].idxmax(axis=1)
df = df.drop(columns=cols_actifs)
df = df.drop(columns=['Etudiants', 'Retraités', 'Inactifs'])


# On remplace les proportions de chaque tranche d'âge par la tranche d'âge majoritaire
cols_ages = ['15-24 ans', '25-39 ans', '40-54 ans', '55-64 ans', '65-79 ans', '80 ans et +']
mask_ages = df[cols_ages].notna().any(axis=1)
df.loc[mask_ages, 'Age majoritaire'] = df.loc[mask_ages, cols_ages].idxmax(axis=1)
df = df.drop(columns=cols_ages)


# On supprime 'famille', déjà divisé en différents types de famille
df = df.drop(columns=['Famille'])

display(df.columns)



# Modélisation

### Split test-train

In [ ]:
X = df.drop(columns=['Code_INSEE', 'Résultat'])
y = df['Résultat']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("✓ Données préparées")
print(f"  Train: {X_train.shape}")
print(f"  Test: {X_test.shape}")

### Fonctions

In [ ]:
groups = ['gauche', 'centre', 'droite']


def train_model_and_detect_overfitting(model):
    model.fit(X_train, y_train)

    y_pred_test = model.predict(X_test)
    y_pred_train = model.predict(X_train)


    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)
    gap = abs(train_score - test_score)


    print(f"  Train : {train_score:.2%}")
    print(f"  Test  : {test_score:.2%}")
    print(f"  Gap   : {gap:.2%}\n")


    if gap > 0.10:
        print("⚠️ OVERFITTING détecté !")
    elif train_score < 0.75:
        print("⚠️ UNDERFITTING détecté !")
    else:
        print("✅ Bon équilibre\n")
    
    return y_pred_test, y_pred_train


def display_confusion_matrix(y, y_pred):
    matrix = confusion_matrix(y, y_pred, labels=groups)

    plt.figure(figsize=(8, 6))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=[f'Prédit {g}' for g in groups],
                yticklabels=[f'Réel {g}' for g in groups])
    plt.title('Matrice de confusion', fontsize=14, pad=15)
    plt.tight_layout()
    plt.show()


def display_metrics(y, y_pred):
    print("\n\nRapport de classification :")
    print(classification_report(y, y_pred, target_names=groups))

## Preprocessor

In [ ]:
num_cols = df.select_dtypes('number').columns
num_cols = num_cols.drop('Résultat')

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OneHotEncoder())
])


preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, ['CSP majoritaire', 'Age majoritaire'])
    ]
)

### Régression logistique + SMOTE

In [ ]:

logistic_smote = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42, sampling_strategy='all')),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])


logistic_smote.fit(X_train, y_train)
lrs_y_pred_test, lrs_y_pred_train = train_model_and_detect_overfitting(logistic_smote)


print('\n---RÉSULTATS DE LA RÉGRESSION LOGISTIQUE + SMOTE SUR LE TEST\n')
display_metrics(y_test, lrs_y_pred_test)


print('\n---RÉSULTATS DE LA RÉGRESSION LOGISTIQUE + SMOTE SUR LE TRAIN\n')
display_metrics(y_train, lrs_y_pred_train)



### Régression logistique + weighted

In [ ]:
logistic_weighted = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000))
])


logistic_weighted.fit(X_train, y_train)
lrw_y_pred_test, lrw_y_pred_train = train_model_and_detect_overfitting(logistic_weighted)


print('\n---RÉSULTATS DE LA RÉGRESSION LOGISTIQUE + WEIGHTED SUR LE TEST\n')
display_metrics(y_test, lrw_y_pred_test)


print('\n---RÉSULTATS DE LA RÉGRESSION LOGISTIQUE + WEIGHTED SUR LE TRAIN\n')
display_metrics(y_train, lrw_y_pred_train)

### XGBoost + Smote

In [ ]:
xgb_smote = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42, sampling_strategy='all')),
    ('classifier', xgb.XGBClassifier(
        n_estimators=200,       # Nombre d'arbres
        max_depth=5,           # Profondeur des arbres
        learning_rate=0.1,      # Taux d'apprentissage (η)
        subsample=0.8,          # Échantillonnage des données
        colsample_bytree=0.8,   # Échantillonnage des features
        reg_alpha=0.1,          # Régularisation L1
        reg_lambda=1.0,         # Régularisation L2
        random_state=42,
        n_jobs=-1,
        objective='multi:softmax',
        eval_metric='mlogloss'   # Métrique d'évaluation
    ))
])


xgb_smote.fit(X_train, y_train)
xgbs_y_pred_test, xgbs_y_pred_train = train_model_and_detect_overfitting(xgb_smote)


print('\n---RÉSULTATS DE XGBOOST + SMOTE SUR LE TEST\n')
display_metrics(y_test, xgbs_y_pred_test)


print('\n---RÉSULTATS DE XGBOOST + SMOTE SUR LE TRAIN\n')
display_metrics(y_train, xgbs_y_pred_train)

### XGBoost + SmoteEEN

In [ ]:
xgb_smoteenn = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTEENN(random_state=42, smote=SMOTE(sampling_strategy='all', random_state=42))),
    ('classifier', xgb.XGBClassifier(
        n_estimators=200,       # Nombre d'arbres
        max_depth=5,           # Profondeur des arbres
        learning_rate=0.1,      # Taux d'apprentissage (η)
        subsample=0.8,          # Échantillonnage des données
        colsample_bytree=0.8,   # Échantillonnage des features
        reg_alpha=0.1,          # Régularisation L1
        reg_lambda=1.0,         # Régularisation L2
        random_state=42,
        n_jobs=-1,
        objective='multi:softmax',
        eval_metric='mlogloss'   # Métrique d'évaluation
    ))
])


xgb_smoteenn.fit(X_train, y_train)
xgbse_y_pred_test, xgbse_y_pred_train = train_model_and_detect_overfitting(xgb_smoteenn)


print('\n---RÉSULTATS DE XGBOOST + SMOTE SUR LE TEST\n')
display_metrics(y_test, xgbse_y_pred_test)


print('\n---RÉSULTATS DE XGBOOST + SMOTE SUR LE TRAIN\n')
display_metrics(y_train, xgbse_y_pred_train)

### XGBoost + Smote Tomek

In [ ]:
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import TomekLinks


xgb_smotetomek = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTETomek(tomek=TomekLinks(sampling_strategy='all'), random_state=42)),
    ('classifier', xgb.XGBClassifier(
        n_estimators=200,       # Nombre d'arbres
        max_depth=5,           # Profondeur des arbres
        learning_rate=0.1,      # Taux d'apprentissage (η)
        subsample=0.8,          # Échantillonnage des données
        colsample_bytree=0.8,   # Échantillonnage des features
        reg_alpha=0.1,          # Régularisation L1
        reg_lambda=1.0,         # Régularisation L2
        random_state=42,
        n_jobs=-1,
        objective='multi:softmax',
        eval_metric='mlogloss'   # Métrique d'évaluation
    ))
])


xgb_smotetomek.fit(X_train, y_train)
xgbst_y_pred_test, xgbst_y_pred_train = train_model_and_detect_overfitting(xgb_smotetomek)


print('\n---RÉSULTATS DE XGBOOST + SMOTE TOMEK SUR LE TEST\n')
display_metrics(y_test, xgbst_y_pred_test)


print('\n---RÉSULTATS DE XGBOOST + SMOTE TOMEK SUR LE TRAIN\n')
display_metrics(y_train, xgbst_y_pred_train)